# Engram — Long-Term Memory for LLM Agents

This notebook shows how to build a simple AI agent that **remembers things about you across conversations** using [engram-ltm](https://pypi.org/project/engram-ltm/).

**What you need:**
- A free [Gemini API key](https://aistudio.google.com/apikey) (1500 calls/day free)
- Nothing else — no local setup, no Ollama, no database

## Step 1 — Install dependencies

In [ ]:
!pip install -q "engram-ltm>=0.2.2" sentence-transformers google-genai

## Step 2 — Add your Gemini API key

In [ ]:
GEMINI_API_KEY = "YOUR_GEMINI_API_KEY_HERE"  # get one free at https://aistudio.google.com/apikey

## Step 3 — Set up Engram

- `embedding_provider="sentence-transformers"` — embeddings run locally in Colab, no API needed
- `llm_provider="gemini"` — Gemini extracts structured facts from your conversations

In [ ]:
from engram import Engram, EngramConfig

memory = Engram(EngramConfig(
    embedding_provider="sentence-transformers",
    llm_provider="gemini",
    llm_model="gemini-2.5-flash",
    gemini_api_key=GEMINI_API_KEY,
))

USER_ID = "demo_user"
print("Engram ready!")

## Step 4 — Build a simple agent

This agent:
1. Retrieves relevant memories before each response
2. Passes memory context + your message to Gemini
3. Stores new facts from each conversation turn

In [ ]:
from google import genai

client = genai.Client(api_key=GEMINI_API_KEY)

def chat(user_message: str) -> str:
    # 1. Retrieve relevant memories
    context = memory.get_context(user_message, user_id=USER_ID)

    # 2. Build prompt with memory context
    system = "You are a helpful personal assistant with memory about the user."
    if context:
        system += f"\n\nWhat you remember about this user:\n{context}"

    prompt = f"{system}\n\nUser: {user_message}\nAssistant:"

    # 3. Generate response
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt,
    )
    answer = response.text.strip()

    # 4. Store this conversation turn in Engram
    memory.add(
        messages=[
            {"role": "user", "content": user_message},
            {"role": "assistant", "content": answer},
        ],
        user_id=USER_ID,
    )

    return answer

print("Agent ready!")

## Step 5 — Talk to the agent

Run each cell one by one. Notice that the agent remembers what you told it earlier — even facts that contradict older ones.

In [ ]:
# Turn 1 — tell the agent something about yourself
response = chat("Hey! I'm Alex. I'm a data science student and I live in Tampa, Florida.")
print(response)

In [ ]:
# Turn 2 — unrelated conversation
response = chat("What are some good Python libraries for data visualization?")
print(response)

In [ ]:
# Turn 3 — update a fact (Engram will supersede Tampa with Austin)
response = chat("Good news — I accepted the offer and I now live in Austin, Texas!")
print(response)

In [ ]:
# Turn 4 — test memory: should say Austin, NOT Tampa
response = chat("What city do I live in?")
print(response)

In [ ]:
# Turn 5 — test memory: should remember name and field of study
response = chat("What do you know about me so far?")
print(response)

## Step 6 — Inspect stored memories

In [ ]:
# See what Engram currently remembers (only active, non-superseded memories)
results = memory.search("user profile", user_id=USER_ID, k=10)
print(f"Active memories ({len(results)} total):\n")
for r in results:
    print(f"  [{r.memory.memory_type.value}] {r.memory.content}")

In [ ]:
# Stats
stats = memory.stats(user_id=USER_ID)
print(f"Total memory records (including superseded): {stats['total_memories']}")